In [5]:
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    DataCollatorForSeq2Seq, TrainingArguments, Trainer
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
from warnings import filterwarnings
filterwarnings('ignore')

# Load and clean dataset
df = pd.read_csv('ak_lectures_summarized 2.csv')
df = df.dropna(subset=['Transcript', 'Summary'])
df['input_text'] = 'summarize: ' + df['Transcript']
df['target_text'] = df['Summary']

In [6]:
df.head()

,Summary,Transcript,input_text,target_text
0,Instead of being localized at a specific organ...,"Unlike most other systems of our body, the im...",summarize: Unlike most other systems of our b...,Instead of being localized at a specific organ...
1,"Leukocytes, or white blood cells, are the cell...","Lucocytes, also known as white blood cells, a...","summarize: Lucocytes, also known as white blo...","Leukocytes, or white blood cells, are the cell..."
2,Our innate immune system consists of non-speci...,Our immune system consists of two divisions. ...,summarize: Our immune system consists of two ...,Our innate immune system consists of non-speci...
3,One way in which our innate immune system deal...,One important aspect of the innate immune sys...,summarize: One important aspect of the innate...,One way in which our innate immune system deal...
4,The innate immune system begins to act immedia...,When one of the many different types of barri...,summarize: When one of the many different typ...,The innate immune system begins to act immedia...


In [7]:
df.shape

(1842, 4)

In [8]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.shape, val_df.shape

((1473, 4), (369, 4))

In [9]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Load tokenizer and model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
def preprocess(example):
    inputs = tokenizer(
        example['input_text'],
        max_length=1024,
        truncation=True,
        padding='max_length'
    )
    targets = tokenizer(
        example['target_text'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs

In [11]:
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

Map:   0%|          | 0/1473 [00:00<?, ? examples/s]

Map:   0%|          | 0/369 [00:00<?, ? examples/s]

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=500,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_dir='./logs',
    fp16=True,
    save_total_limit=2,
    report_to='none',
    load_best_model_at_end=True,
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [14]:
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,No log,2.006266
2,No log,1.937762
3,No log,1.915248
4,No log,1.898565
5,No log,1.889779
6,2.014000,1.880163
7,2.014000,1.883215
8,2.014000,1.878569


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=744, training_loss=1.9218403805968582, metrics={'train_runtime': 733.195, 'train_samples_per_second': 16.072, 'train_steps_per_second': 1.015, 'total_flos': 3189735577092096.0, 'train_loss': 1.9218403805968582, 'epoch': 8.0})

In [15]:
results = trainer.evaluate()
print("Final Evaluation Results:", results)

Final Evaluation Results: {'eval_loss': 1.8785687685012817, 'eval_runtime': 8.0378, 'eval_samples_per_second': 45.908, 'eval_steps_per_second': 2.986, 'epoch': 8.0}


In [23]:
model.save_pretrained('./t5_finetuned')
tokenizer.save_pretrained('./t5_finetuned')

('./t5_finetuned/tokenizer_config.json',
 './t5_finetuned/special_tokens_map.json',
 './t5_finetuned/spiece.model',
 './t5_finetuned/added_tokens.json')

In [24]:
!zip -r saved_model.zip t5_finetuned

  adding: t5_finetuned/ (stored 0%)
  adding: t5_finetuned/config.json (deflated 63%)
  adding: t5_finetuned/generation_config.json (deflated 29%)
  adding: t5_finetuned/tokenizer_config.json (deflated 94%)
  adding: t5_finetuned/spiece.model (deflated 48%)
  adding: t5_finetuned/model.safetensors (deflated 8%)
  adding: t5_finetuned/added_tokens.json (deflated 83%)
  adding: t5_finetuned/special_tokens_map.json (deflated 85%)


In [26]:
from google.colab import files
files.download('saved_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>